# 01. Exploratory Data Analysis (EDA) — Telco Customer Churn
**Author:** Farid Masood Khan  
**Objective:** In-depth exploratory data analysis of the IBM Telco Customer Churn dataset.  
**Curriculum Alignment:** DeepLearning.AI / Stanford Online Machine Learning Specialization (Supervised Learning, Regression and Classification).

---
### Research Questions:
1. What is the overall customer churn rate and class imbalance ratio?
2. How does churn probability vary across contract types, tenure cohorts, and internet service types?
3. Are there data quality issues or missing values that require domain-specific imputation?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')


## 1. Data Ingestion & Schema Inspection

In [ ]:
data_path = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 2. Missing Value & Anomaly Audit
In the raw IBM dataset, `TotalCharges` is stored as an object. When new customers have `tenure == 0`, their `TotalCharges` field contains whitespace characters (`" "`).


In [ ]:
# Audit blank TotalCharges strings
blank_charges = df[df['TotalCharges'].astype(str).str.strip() == '']
print(f"Number of rows with whitespace TotalCharges: {len(blank_charges)}")
print("Tenure values for these rows:", blank_charges['tenure'].unique())

# Fix: parse numeric, impute tenure==0 with 0.0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
print(f"Missing values remaining in TotalCharges: {df['TotalCharges'].isna().sum()}")


## 3. Target Distribution: Class Imbalance
The target variable is `Churn` (`Yes` / `No`). Let us quantify the distribution.


In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate = df['Churn'].value_counts(normalize=True)

print("Churn Counts:
", churn_counts)
print("
Churn Proportion:
", churn_rate)

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
sns.barplot(x=churn_rate.index, y=churn_rate.values * 100, palette=['#10b981', '#ef4444'], ax=ax)
ax.set_ylabel("Percentage (%)", fontweight='bold')
ax.set_title(f"Target Distribution: {churn_rate['Yes']:.1%} Churn Rate", fontweight='bold')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()


## 4. Bivariate Analysis: Churn by Contract Type & Tenure
Hypothesis: Month-to-month contracts and customers within their first year experience disproportionately high churn.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

# Churn by Contract
contract_churn = df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
sns.barplot(data=contract_churn, x='Contract', y='Churn', palette='mako', ax=ax1)
ax1.set_title("Churn Rate by Contract Type", fontweight='bold')
ax1.set_ylabel("Churn Rate (%)", fontweight='bold')
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 5),
                ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Tenure distribution by Churn status
sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, palette=['#10b981', '#ef4444'], ax=ax2)
ax2.set_title("Tenure Density: Churners vs. Retained", fontweight='bold')
ax2.set_xlabel("Tenure (Months)", fontweight='bold')
plt.tight_layout()
plt.show()


## 5. EDA Summary & Modeling Takeaways
1. **Severe Contract Disparity:** Month-to-month contracts exhibit ~42.7% churn, whereas two-year contracts exhibit under ~3% churn.
2. **Hazard Period:** Churn risk is heavily concentrated in months 0–12.
3. **Class Imbalance:** Churners represent 26.5% of samples; models must use balanced class weights or focus on Precision-Recall / ROC-AUC rather than raw accuracy.
4. **Data Hygiene Rule:** `customerID` must be discarded prior to feature encoding; all scalers and encoders must be fit exclusively on training data.
